# 🎙️ VoiceBatch Studio v0.0 [Generation Fix]
सिर्फ 'ऑडियो बनाएं' बटन को फिक्स किया गया है। बाकी सभी फीचर्स लॉक हैं।

In [ ]:
# @title 🛠️ Step 1: सेटअप
import os
from google.colab import drive
print("⏳ सेटअप हो रहा है...")
!pip install -q coqpit-config coqui-tts gradio librosa soundfile
if not os.path.exists('/content/drive'): drive.mount('/content/drive')
os.makedirs("outputs", exist_ok=True)
print("✅ ड्राइव कनेक्ट हो गई!")

In [ ]:
# @title 🚀 Step 2: हाई-स्पीड स्टूडियो (Fixed Button)
import gradio as gr
import torch, librosa, re, numpy as np, soundfile as sf
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_path = "/content/drive/MyDrive/VoiceBatchModels/"
tts = TTS(model_path=model_path, config_path=model_path + "config.json").to(device)

def fixed_turbo_engine(text, audio_sample, silence_rem):
    if not audio_sample or not text: return None
    try:
        with torch.inference_mode():
            parts = re.split(r'(?<=[।?!])\s+', text)
            combined = []
            for p in parts:
                if len(p.strip()) < 2: continue
                wav = tts.tts(text=p, speaker_wav=audio_sample, language='hi')
                combined.append(np.array(wav))
            
            final_output = np.concatenate(combined)
            
            # Silence Remover का नाम फिक्स किया गया
            if silence_rem:
                final_output, _ = librosa.effects.trim(final_output, top_db=20)
            
            out_path = "outputs/v0_fixed_result.wav"
            sf.write(out_path, final_output, 24000)
            return out_path
    except Exception as e:
        print(f"❌ एरर: {str(e)}")
        return None

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ VoiceBatch Studio v0.0")
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label="स्क्रिप्ट (हिंदी)", lines=10)
            cnt = gr.Markdown("Shabd: 0")
            txt.change(lambda x: f"Shabd: {len(x.split())}", inputs=[txt], outputs=[cnt])
            smp = gr.Audio(label="नमूना अपलोड करें", type='filepath')
            sil = gr.Checkbox(label="साइलेंस रिमूवर", value=True)
            btn = gr.Button("ऑडियो बनाएं ⚡", variant="primary")
        with gr.Column():
            out = gr.Audio(label="परिणाम डाउनलोड करें")

    btn.click(fixed_turbo_engine, [txt, smp, sil], out)

demo.queue().launch(share=True, debug=True)